### 1- Load the raw data

In [94]:
from pathlib import Path
import json

import pandas as pd


RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

nodes = pd.read_csv(
    RAW_DATA_DIR / "isebel-mecklenburg-nodes.csv"
)

edges = pd.read_csv(
    RAW_DATA_DIR / "isebel-mecklenburg-edges.csv"
)

print("Nodes:", nodes.shape)
print("Edges:", edges.shape)

Nodes: (20005, 3)
Edges: (40604, 4)


### 2- Extract keyword names

In [95]:
def get_keyword_name(properties):
    properties = json.loads(properties)
    return str(properties.get("name", "")).strip()


keywords = nodes[nodes["label"] == "keyword"].copy()

keywords["keyword"] = keywords["properties"].apply(
    get_keyword_name
)

content_edges = (
    edges.loc[
        edges["label"] == "content",
        ["src-id", "dst-id"]
    ]
    .rename(columns={
        "src-id": "story_id",
        "dst-id": "keyword_id"
    })
    .drop_duplicates()
)

keyword_names = keywords.set_index("id")["keyword"]

print("Keywords:", len(keywords))
print("Story-keyword relationships:", len(content_edges))

Keywords: 2439
Story-keyword relationships: 30410


### 3- Find keywords attached to identical story sets

In [96]:
keyword_story_sets = (
    content_edges
    .groupby("keyword_id")["story_id"]
    .apply(frozenset)
)

groups_by_stories = {}

for keyword_id, story_set in keyword_story_sets.items():
    groups_by_stories.setdefault(
        story_set,
        []
    ).append(keyword_id)


candidate_groups = [
    keyword_ids
    for story_set, keyword_ids in groups_by_stories.items()
    if len(keyword_ids) > 1 and len(story_set) >= 2
]


candidate_rows = []

for group_id, keyword_ids in enumerate(
    candidate_groups,
    start=1
):
    for keyword_id in keyword_ids:
        candidate_rows.append({
            "group_id": group_id,
            "keyword_id": keyword_id,
            "keyword": keyword_names.get(keyword_id)
        })

candidate_table = pd.DataFrame(candidate_rows)

display(candidate_table)
print("Candidate groups:", candidate_table["group_id"].nunique())

,group_id,keyword_id,keyword
0,1,4,Historische Sagen
1,1,6,historical legends
2,2,5,Dänen
3,2,7,Danish people
4,3,16,Frevelsage
...,...,...,...
701,287,18333,"Persons: Luther, Goethe etc."
702,287,18334,Goethe
703,287,18335,"Luther, Martin"
704,288,18974,Ei


Candidate groups: 288


### 4- Create the canonical keyword decisions

In [97]:
canonical_labels = {
    1: "historical legends",
    2: "Danish people",
    3: "sacrilege",
    4: "witches",
    5: "on cattle",
    6: "Knight varia",
    7: "local legends",
    8: "Blocksberg varia",
    9: "Horse shoe etc.",
    10: "local legends: varia",
    11: "werewolf",
    12: "dead",
    13: "Something on the head",
    14: "from the french period",
    15: "French people",
    16: "When",
    17: "The shot werewolf",
    18: "The ample meal",
    19: "local legends: churches and passages",
    20: "Historic, elderly",
    21: "Kette tragen, Ring",
    22: "Self-transformation into the wolf form by means of magic",
    23: "werfen Steine",
    24: "giants",
    25: "Wolf und Fuchs auf der Hochzeit, Kranker trägt Gesunden",
    26: "bells",
    27: "Robber varia",
    28: "haunting animals",
    29: "treasure",
    30: "goldene Wiege",
    31: "Totenmesse",
    32: "Seejungfer",
    33: "Persönlichkeiten, Familie etc.",
    34: "Krieg, siebenjähriger",
    35: "Taters, krup unner",
    36: "destruction",
    37: "Towns",
    38: "Wiederkehr, Totenhemd",
    39: "Treasure hunter",
    40: "Tatern (gypsies)",
    41: "Literatur",
    42: "See, Opfer, Varia",
    43: "moort",
    44: "Taufe, Lattenstiger",
    45: "Zwergensage",
    46: "to purge the bowels",
    47: "Ewig Leben: I",
    48: "Ewiger Jude",
    49: "kaufen Korn",
    50: "Toys",
    51: "An Menschen",
    52: "Mountain: singing, thunder etc.",
    53: "Kenntnis von den Wenden, meckl.",
    54: "Blutfleck",
    55: "wissen",
    56: "Blutstropfen etc.",
    57: "weiblich: Deichsel",
    58: "ghosts",
    59: "Robber-knight gen.",
    60: "Pferd, allg. Abwehr etc.",
    61: "Tote red., erste Frau, Mutter",
    62: "Mountain, non movere",
    63: "Mountain open",
    64: "Buttern",
    65: "Kirche",
    66: "French people etc., search town",
    67: "devil's confederate",
    68: "Pumpfuß, Columbus",
    69: "Buttern, Lappen etc.",
    70: "30-/100-year war, blending",
    71: "Verwandlung in verschiedene Tiere",
    72: "Unschuldig: Taube, Torgelow etc.",
    73: "Werwolf Varia",
    74: "unterirdischer Gang",
    75: "Brüder",
    76: "Buttern, Melken",
    77: "Hexenhalfter",
    78: "wo, Aufenthaltsort",
    79: "Schatz, wo",
    80: "Burg, Schloss, Signal",
    81: "Untergang: Lit.",
    82: "Hufeisen, verkehrt",
    83: "Esche",
    84: "Grausame Herren",
    85: "Verwandlung in einen Bären",
    86: "Berg nicht fortschaffen",
    87: "Berg: heilige Berge; Zwerge",
    88: "ablassen, See ausmessen etc.",
    89: "Steinsagen Varia (m) s",
    90: "Steinsage",
    91: "weiblich: Namen",
    92: "Tod varia",
    93: "Gen.: Skeleton found",
    94: "Untergang: Prophezeiung, cf Wasser",
    95: "Teufel und Advokat",
    96: "noble family",
    97: "Lattenspringer (moonstruck)",
    98: "Abzug etc.",
    99: "Harte Herrin",
    100: "burried an old",
    101: "not to fear",
    102: "Russian people",
    103: "Tribunal mountain, judged inculpable",
    104: "Taube varia",
    105: "Mountains",
    106: "entrückt: Paradies",
    107: "Nest wird ausgehoben - Schätze",
    108: "Werwolf im Haus",
    109: "Names",
    110: "Ort, Orte, wo Räuber hausen",
    111: "Hexen erkennen",
    112: "Pale girl",
    113: "Worauf",
    114: "Teufel varia",
    115: "Elster lehrt Taube Nest bauen",
    116: "Ghost",
    117: "Werber",
    118: "Schürze platzt",
    119: "Scheintoter",
    120: "Entheiligung des Feiertages, Karfreitag",
    121: "grundlos, ausmessen etc.",
    122: "abziehen",
    123: "einmauern",
    124: "Names I",
    125: "to bake etc.",
    126: "waschen, sich",
    127: "Körperteile, Rippe etc.",
    128: "Fischersagen",
    129: "Name: varia",
    130: "erwürgen",
    131: "Mekelborger",
    132: "Berge hohl",
    133: "Gang, Lit.",
    134: "Edelmann, Toter, Geld",
    135: "goldener Sarg",
    136: "Großherzöge etc.",
    137: "Botschaft vom Jenseits",
    138: "Mäuseturm",
    139: "Liebeszauber: Silvester",
    140: "heben, quis",
    141: "Teufel baut, Lit.",
    142: "Wechselbalg I",
    143: "Kyffhäuser, varia",
    144: "Menschenkeule",
    145: "Maränen",
    146: "Witch general",
    147: "Mordkrug etc.",
    148: "Starke Leute",
    149: "Katze, Ratte etc.",
    150: "gut",
    151: "Totentrauung",
    152: "Gegenstände, Geräte Varia",
    153: "Totenkopf etc.",
    154: "Grab etc.",
    155: "wie See entstanden etc.",
    156: "Ochse, Bolle",
    157: "Häkel etc.",
    158: "Berg vom Riesen zusammengetragen",
    159: "Butterhex",
    160: "The husband as werewolf",
    161: "Tritt",
    162: "Kirchbau",
    163: "Ritternamen",
    164: "Dörfer",
    165: "Glaube",
    166: "Viting /raubt Mädchen - 7 Söhne - verraten",
    167: "Werwolf fällt einen Menschen an",
    168: "Victim, last hour has come",
    169: "Friede in der Welt",
    170: "Wenden: Schlacht, Heiden",
    171: "to carry mountains etc.",
    172: "Werwolf als Viehdieb",
    173: "Pest, allg.",
    174: "Schlaraffenland",
    175: "Hand aus Grab",
    176: "Toad etc.",
    177: "Röpke etc.",
    178: "Geld brennt",
    179: "to build",
    180: "weiblich: Spinnen",
    181: "Hemd etc.",
    182: "Fische im Eis angeln",
    183: "Blocksberg Orte",
    184: "Seeräuber Varia",
    185: "kennenlernen",
    186: "Schatz, Wert",
    187: "Tiere: Sagen, Märchen etc.",
    188: "Wölfe zum Wallach gemacht",
    189: "Streiche",
    190: "Seele als Maus",
    191: "tragen Berge zusammen",
    192: "entrückt: Varia",
    193: "Tote: unverweslich",
    194: "to see leftovers",
    195: "Mönche sehen nach etc.",
    196: "Wassermann",
    197: "Namenswechsel",
    198: "jagt Frauen etc.",
    199: "Teufel, Namen a",
    200: "wie greifen",
    201: "Tod fordert, Galgen, Zauberer, varia",
    202: "keine Ruhe im Grab",
    203: "regnen",
    204: "Mäuse machen etc.",
    205: "Blocksberg Mahl",
    206: "Witching dinner",
    207: "Schinnerhannes",
    208: "Geldbrennen: Mädchen holt Kohle von Backofen",
    209: "Stroh etc.",
    210: "Etmylogie Lit.",
    211: "Puckel",
    212: "Kyffhäuser: Kaiser im Berg",
    213: "lindworm",
    214: "Schlacht etc.",
    215: "to invite etc.",
    216: "Meineid",
    217: "Namenerklärung",
    218: "Knothole",
    219: "Geldfeuer: Warnung nicht zum dritten Mal zu kommen",
    220: "Tote: Hand heraus etc.",
    221: "Bach pflügen",
    222: "Kreuz (sakral)",
    223: "Storch: geschossen - hinterher verletzter Mensch",
    224: "fesseln",
    225: "Schellfisch, Placken",
    226: "Tier- und Pflanzenlegenden",
    227: "Storch: große Mauer, Läversee",
    228: "Tote: Kirche, Messe",
    229: "Räuberbanden",
    230: "demons",
    231: "Juden, Hostie",
    232: "Wettermachen",
    233: "weiblich: horn",
    234: "Moon",
    235: "Hunde, Brot, Bier",
    236: "Semp",
    237: "Feldmaus und Stadtmaus",
    238: "Brombeerranke",
    239: "Waul: ewig Jagen",
    240: "to drain etc., to measure",
    241: "Aalbluser etc.",
    242: "Erklärungen etc.",
    243: "Pferdestall etc.",
    244: "Spruch",
    245: "zwei",
    246: "Teich, auswaschen, fällen, Baum",
    247: "Receipt, money under head",
    248: "Chimken etc.",
    249: "verwünscht",
    250: "Spuk: dämonisch",
    251: "Bergnamen",
    252: "Ortssagen (außer Bornholm)",
    253: "preußische Könige",
    254: "zeigt sich",
    255: "Stein, Spuren (m) s",
    256: "Geschirr (Haushalt)",
    257: "Vow",
    258: "Kröte, Gevatter",
    259: "Trick",
    260: "buffalo's head, coat of arms",
    261: "Teufelsbündner: Pferde festmachen",
    262: "Bäume, auf Hunden, Sense, Tannen",
    263: "Gründung",
    264: "Keule",
    265: "Freischütz, varia Lit. I",
    266: "Kräwter etc.",
    267: "Butt - schiefes Maul",
    268: "Blocksberg treiben",
    269: "wie: Art der Erscheinung, Knäuel, Feuer, Ross etc.",
    270: "Viting, lit.",
    271: "Schatz: Geldbrennen",
    272: "Mineral spring / holy spring",
    273: "Zauberbuch, zitieren",
    274: "The werewolf as rustler",
    275: "Biss",
    276: "Mountain, open",
    277: "Wolfsriemen",
    278: "Verwandlung in einen Hund",
    279: "auspeitschen",
    280: "Topographisches",
    281: "schenken",
    282: "A 80447",
    283: "Teufelsbündner: Lit. varia",
    284: "Vorstadt",
    285: "Dänen, Russen etc.",
    286: "Schäfer II",
    287: "Persons: Luther, Goethe etc.",
    288: "Ei",
}

print("Groups selected for merging:", len(canonical_labels))

Groups selected for merging: 288


### 5- Create the keyword-to-canonical mapping

In [98]:
keyword_mapping = {
    keyword_id: {
        "keyword_id": keyword_id,
        "keyword": keyword_names.get(keyword_id)
    }
    for keyword_id in keywords["id"]
}


for group_id, canonical_keyword in canonical_labels.items():
    group_rows = candidate_table[
        candidate_table["group_id"] == group_id
    ]

    if group_rows.empty:
        print(f"Warning: group {group_id} does not exist")
        continue

    group_keyword_ids = group_rows["keyword_id"].tolist()

    # Use the first keyword ID as the shared canonical ID.
    canonical_keyword_id = group_keyword_ids[0]

    for keyword_id in group_keyword_ids:
        keyword_mapping[keyword_id] = {
            "keyword_id": canonical_keyword_id,
            "keyword": canonical_keyword
        }

In [99]:
clean_content_edges = content_edges.copy()

clean_content_edges["clean_keyword_id"] = (
    clean_content_edges["keyword_id"]
    .map(lambda keyword_id: keyword_mapping[keyword_id]["keyword_id"])
)

clean_content_edges["keyword"] = (
    clean_content_edges["keyword_id"]
    .map(lambda keyword_id: keyword_mapping[keyword_id]["keyword"])
)

clean_content_edges = clean_content_edges[
    [
        "story_id",
        "clean_keyword_id",
        "keyword"
    ]
].rename(columns={
    "clean_keyword_id": "keyword_id"
})

### 6- Remove duplicate story-keyword relationships

In [100]:
edges_before = len(clean_content_edges)

clean_content_edges = (
    clean_content_edges
    .drop_duplicates(
        subset=["story_id", "keyword_id"]
    )
    .reset_index(drop=True)
)

edges_after = len(clean_content_edges)

print("Edges before cleaning:", len(content_edges))
print("Edges after cleaning:", edges_after)
print("Duplicate edges removed:", edges_before - edges_after)

print(
    "Stories before cleaning:",
    content_edges["story_id"].nunique()
)

print(
    "Stories after cleaning:",
    clean_content_edges["story_id"].nunique()
)

print(
    "Keywords before cleaning:",
    content_edges["keyword_id"].nunique()
)

print(
    "Keywords after cleaning:",
    clean_content_edges["keyword_id"].nunique()
)

Edges before cleaning: 30410
Edges after cleaning: 20720
Duplicate edges removed: 9690
Stories before cleaning: 4577
Stories after cleaning: 4577
Keywords before cleaning: 2439
Keywords after cleaning: 2021


### 7- Save the processed keyword data

In [101]:
output_path = (
    PROCESSED_DATA_DIR
    / "clean_story_keyword_edges.csv"
)

clean_content_edges.to_csv(
    output_path,
    index=False
)

print("Saved cleaned data to:")
print(output_path)

Saved cleaned data to:
..\data\processed\clean_story_keyword_edges.csv
